In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt 
import math

torch.set_default_dtype(torch.float64)


In [2]:

# define Net()
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(d_in, nodes))  # First layer
        for _ in range(1, layers):  # Remaining hidden layers
            self.layers.append(nn.Linear(nodes, nodes))
        self.out = nn.Linear(nodes, d_out)
    def forward(self, x):
        for layer in self.layers:
            x = torch.sigmoid(layer(x))  # Or use F.relu if preferred
        return self.out(x)

# build the NN
def build_nn():
    model = Net()
    optimizer = optim.AdamW(model.parameters(), lr=lr, eps=eps,
        betas=betas,weight_decay=wd)
    return model, optimizer

# rebuild the NN from (model_dict, optimizer_dict) 
def rebuild_nn():
    model = Net()
    optimizer = optim.AdamW(model.parameters())
    if 'model_dict' in globals():
        model.load_state_dict(model_dict)
    else:
        print('**** Not enough info to recover the model ****')
    if 'optimizer_dict' in globals():
        optimizer.load_state_dict(optimizer_dict)
    else:
        print('**** Not enough info to recover the optimizer ****')
    return model, optimizer

# print parameters, model, optimer
def print_params():
    print('Structure of neural network: ',model) # print the model
    print('Optimizer:  ',optimizer)
    print(f'parameters: d_in = {d_in}, d_out = {d_out}, '\
         f'layers = {layers}, nodes = {nodes}')
    print(f'    tol = {tol:.4e}, epochs = {epochs}, seg_size = {seg_size}, epoch_0 = {epoch_0}')
    print(f'    N_train = {N}, N_test = {N_test}')
    print(f'    output file: {out_file:s}\n')


In [3]:
# train the NN
def train_nn():
    tolc = tol # tolc is the current tolerance 
    loss_curr_min=1.0e4 # the current minimum loss
    epochs_soft = epochs-2*seg_size # soft cap on the number of epochs
        # after reaching epochs_soft, stop when loss <= loss_curr_min
    epochs_hard = epochs+40*seg_size
    loss_v = np.zeros(epochs_hard, dtype=float) # for training loss
    loss_test_v = np.zeros(epochs_hard, dtype=float) # for test loss
    # loss_v[i] contains the loss AFTER i epochs are COMPLETED (at the 
    # BEGINNING of the epoch with index i    
    #
    model.train(); start = time.time()
    for i in range(epochs_hard):
        optimizer.zero_grad() # zero the gradients
        loss = F.mse_loss(model(xtrain),ytrain) # train loss
        with torch.no_grad(): # no need for grad in loss_test!!!
            #loss_test = F.mse_loss(model(xtest),ytest) # test loss
            loss_v[i] = loss.item()
        #loss_test_v[i] = loss_test.item()
        #
        if (i+epoch_0)%seg_size == 0:
            loss_curr_min = min([np.min(loss_v[max(0, i+1-seg_size):i+1]),
                loss_curr_min]) # update loss_curr_min
            tolc = loss_curr_min if i>=epochs_soft else tol # revise tolc at i >= epochs_soft
            print(f'i = {i+epoch_0}, loss = {loss.item():.4e} '
                  f'loss_curr_min = {loss_curr_min:.4e}')
            current=time.time()
            print(f'Elapsed time = {current-start:.2f}s,  CA time = {time.ctime(current):s}')
        if loss.item() <= 1.4*tolc:
            loss_v=loss_v[0:i+1]
            #loss_test_v=loss_test_v[0:i+1]
            break
        loss.backward() # Back propagation to compute all gradients
        optimizer.step() # optimizer step (updating)
    # print more info after training
    current = time.time();
    msg_train=[]
    msg_train.append(f'Training info: tol_actual = {tolc:.4e}, '\
            f'epochs_actual = {epoch_0+loss_v.size-1}')
    msg_train.append(f'    loss_train(MSE) = {loss_v[-1]:.4e}')
    msg_train.append(f'    Elapsed time = {current-start:.2f}s,  '\
            f'CA time = {time.ctime(current):s}')
    return {'loss_v':loss_v, 'msg_train':msg_train}

# save model and key variables to a data file
def save_model_vars():
    np_vars = {name: val for name, val in globals().items()
            if isinstance(val, (float, int))}
    np_vars['model_dict'] = model.state_dict()
    np_vars['optimizer_dict'] = optimizer.state_dict()
    np_vars['epoch_0']=epoch_0+len(loss_v)-1
    #np_vars['t_train_elapsed'] = t_train_elapsed
    #np_vars['msg_train']=msg_train; np_vars['msg_test']=msg_test
    torch.save(np_vars, out_file)
    print(np_vars.keys())


In [4]:
# test the trained model and plot figures
def test_and_plotting():
    # calculate and print training loss and testing loss (MSE)
    model.eval()
    with torch.no_grad():
        err_train=(model(xtrain)-ytrain).cpu().numpy()
            #
    msg_test = []
    msg_test.append(f'    rms_err_train = {np.sqrt(np.mean(err_train**2)):.4e}')
    msg_test.append(f'    max_err_train = {np.max(np.abs(err_train)):.4e}')
    #
    # plot training loss (MSE) vs epoch
    plt.figure(figsize=(4.8,3.6))
    plt.semilogy(np.arange(len(loss_v))/1000, loss_v,'b-')
    plt.xlabel('Epochs (in 1000s)')
    plt.ylabel('MSE of training')
    plt.xticks(rotation=0, ha='center')
    plt.title('Loss (MSE) vs epochs ')
    plt.savefig('fig_1.pdf',bbox_inches='tight')
    plt.grid(True)
    plt.show()
    
    plt.figure(figsize=(4.8,3.6))
    plt.hist(err_train, bins=200)
    plt.xlabel('Point-wise traning error')
    plt.ylabel('Count')
    plt.xticks(rotation=0, ha='center')
    plt.title('Distribution of point-wise error')
    plt.savefig('fig_2.pdf',bbox_inches='tight')
    plt.grid(True)
    plt.show()
    return {'err_train':err_train}
